<a href="https://colab.research.google.com/github/gracemaria321/AI-for-CyberSecurity/blob/main/LLMs_for_vul_detection_Lab1_Part_I.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [1]:
!pip install transformers torch datasets huggingface_hub


In [2]:
from datasets import load_dataset

# Load CodeXGLUE vulnerability detection dataset
dataset = load_dataset("code_x_glue_cc_defect_detection", split="train")

# View sample data
print(dataset[0])


/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:94: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


README.md: 0.00B [00:00, ?B/s]

data/train-00000-of-00001.parquet:   0%|          | 0.00/17.8M [00:00<?, ?B/s]

data/validation-00000-of-00001.parquet:   0%|          | 0.00/2.21M [00:00<?, ?B/s]

data/test-00000-of-00001.parquet:   0%|          | 0.00/2.23M [00:00<?, ?B/s]

Generating train split:   0%|          | 0/21854 [00:00<?, ? examples/s]

Generating validation split:   0%|          | 0/2732 [00:00<?, ? examples/s]

Generating test split:   0%|          | 0/2732 [00:00<?, ? examples/s]

{'id': 0, 'func': 'static av_cold int vdadec_init(AVCodecContext *avctx)\n\n{\n\n    VDADecoderContext *ctx = avctx->priv_data;\n\n    struct vda_context *vda_ctx = &ctx->vda_ctx;\n\n    OSStatus status;\n\n    int ret;\n\n\n\n    ctx->h264_initialized = 0;\n\n\n\n    /* init pix_fmts of codec */\n\n    if (!ff_h264_vda_decoder.pix_fmts) {\n\n        if (kCFCoreFoundationVersionNumber < kCFCoreFoundationVersionNumber10_7)\n\n            ff_h264_vda_decoder.pix_fmts = vda_pixfmts_prior_10_7;\n\n        else\n\n            ff_h264_vda_decoder.pix_fmts = vda_pixfmts;\n\n    }\n\n\n\n    /* init vda */\n\n    memset(vda_ctx, 0, sizeof(struct vda_context));\n\n    vda_ctx->width = avctx->width;\n\n    vda_ctx->height = avctx->height;\n\n    vda_ctx->format = \'avc1\';\n\n    vda_ctx->use_sync_decoding = 1;\n\n    vda_ctx->use_ref_buffer = 1;\n\n    ctx->pix_fmt = avctx->get_format(avctx, avctx->codec->pix_fmts);\n\n    switch (ctx->pix_fmt) {\n\n    case AV_PIX_FMT_UYVY422:\n\n        vda_c

In [3]:
from transformers import AutoTokenizer, AutoModelForSequenceClassification
import torch

# Load the correct CodeBERT model for classification
MODEL_NAME = "microsoft/codebert-base"
tokenizer = AutoTokenizer.from_pretrained(MODEL_NAME)
model = AutoModelForSequenceClassification.from_pretrained("microsoft/codebert-base", num_labels=2)  # 2 labels: Safe, Vulnerable

# Function to detect vulnerabilities
def detect_vulnerabilities(code_snippet):
    inputs = tokenizer(code_snippet, return_tensors="pt", truncation=True, padding=True, max_length=512)
    with torch.no_grad():
        outputs = model(**inputs)
        probs = torch.nn.functional.softmax(outputs.logits, dim=-1)

    return {"Safe Code": probs[0][0].item(), "Vulnerable Code": probs[0][1].item()}

# ✅ **Use this for user input**
user_code = input("Enter your code snippet:\n")
result = detect_vulnerabilities(user_code)
print("\nVulnerability Scores:", result)



config.json:   0%|          | 0.00/498 [00:00<?, ?B/s]

tokenizer_config.json:   0%|          | 0.00/25.0 [00:00<?, ?B/s]

vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

special_tokens_map.json:   0%|          | 0.00/150 [00:00<?, ?B/s]

pytorch_model.bin:   0%|          | 0.00/499M [00:00<?, ?B/s]

model.safetensors:   0%|          | 0.00/499M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/197 [00:00<?, ?it/s]

RobertaForSequenceClassification LOAD REPORT from: microsoft/codebert-base
Key                        | Status     | 
---------------------------+------------+-
pooler.dense.weight        | UNEXPECTED | 
pooler.dense.bias          | UNEXPECTED | 
classifier.dense.weight    | MISSING    | 
classifier.dense.bias      | MISSING    | 
classifier.out_proj.bias   | MISSING    | 
classifier.out_proj.weight | MISSING    | 

Notes:
- UNEXPECTED	:can be ignored when loading from different task/architecture; not ok if you expect identical arch.
- MISSING	:those params were newly initialized because missing from the checkpoint. Consider training on your downstream task.


Enter your code snippet:
import sqlite3, hashlib  def  login(username, password):      conn = sqlite3.connect('users.db')      query = f"SELECT * FROM users WHERE name='{username}' AND pwd='{password}'"      result = conn.execute(query).fetchone()      if result:          token = hashlib.md5(username.encode()).hexdigest()          print(f"Login OK. Token: {token}")          return token      return None

Vulnerability Scores: {'Safe Code': 0.3854092061519623, 'Vulnerable Code': 0.6145907640457153}


In [5]:
user_code = input("Enter your code snippet: ")
result = detect_vulnerabilities(user_code)
print(result)



Enter your code snippet: import tempfile, os TICKET_PRICE = 29999  # fits in uint16 (max 65535) MAX_SEATS = 200 def reserve_seat(user_id, num_tickets):     seats_left = get_available_seats()  # BUG: no lock — race condition here     if seats_left >= num_tickets:         charge = TICKET_PRICE * num_tickets  # BUG: uint16 overflow if num_tickets >= 3         tmp = tempfile.mktemp()  # BUG: TOCTOU — another process can grab this path first         with open(tmp, 'w') as f:             f.write(f'booking:{user_id}:{charge}')         commit_booking(tmp)         decrement_seats(num_tickets)  # BUG: also non-atomic
{'Safe Code': 0.388431578874588, 'Vulnerable Code': 0.6115683913230896}
